# SVG Drawing Module

This notebook demonstrates the new SVG-based drawing and visualization capabilities introduced in **v1.5** of the library.

## Why SVG?

SVG (Scalable Vector Graphics) is an ideal format for mathematical visualizations because it:

- **Scales perfectly** – No pixelation at any zoom level
- **Editable** – Can be refined in vector graphics editors like Inkscape or Adobe Illustrator
- **Web-native** – Renders directly in browsers and Jupyter notebooks
- **Code-based** – Maintains an explicit, inspectable representation
- **Layered** – Elements can be manipulated, reordered, and styled individually

### Available Elements

The library provides the following predefined elements for the **Poincaré disk model**:

- **`Polygon`** – Hyperbolic polygons with geodesic edges
- **`Tiling`** – Collections of polygons forming complete tilings
- **`Geodesic`** – Geodesic arcs (hyperbolic "straight lines")
- **`HyperbolicCircle`** – Circles at constant hyperbolic distance from a center
- **`Horocycle`** – Limiting circles tangent to the boundary
- **`UnitCircle`** – The domain boundary (unit disk)

Additional elements can be implemented straightforwardly by extending the `SvgElement` base class.

For convenient composition, we provide an **`SVGCanvas`** class that:

- **Collects** multiple `SVGElement` objects
- **Manages** the coordinate system and viewport
- **Renders** the complete SVG document
- Supports **manipulation** of elements before final rendering

Elements can be added, removed, and modified at any time before calling `canvas.render()`.
The following examples demonstrate practical usage patterns, from basic shapes to complex tilings with custom styling.


## Basic Example: Tiling with Inscribed Circles
This example creates a {5,4} tiling and draws the inscribed circle for each polygon.


In [ ]:
from hypertiling import HyperbolicTiling
from hypertiling.graphics.svg import Tiling, HyperbolicCircle, UnitCircle, SVGCanvas, draw_svg
from hypertiling.util import inradius_regular_polygon

# Create tiling
T = HyperbolicTiling(5, 4, 5) # last parameter is the number of layers

# Create SVG canvas
canvas = SVGCanvas()

# Add boundary and tiling
canvas.add(UnitCircle())
canvas.add(Tiling(T, lw=0.3, skip_first=True))

# Add inscribed circles for each polygon
radius = inradius_regular_polygon(5,4)
circles = [HyperbolicCircle(z0=complex(T[i][0]), R=radius, lw=0.6) for i in range(len(T))]
canvas.add(*circles)

# Render and display
draw_svg(canvas.render())

The canvas allows to adjust the SVG viewport

In [ ]:
from hypertiling import HyperbolicTiling
from hypertiling.graphics.svg import Tiling, HyperbolicCircle, UnitCircle, SVGCanvas, draw_svg
from hypertiling.util import outradius_regular_polygon

# Create tiling
T = HyperbolicTiling(5, 4, 7) # last parameter is the number of layers

# height and width control the size of the final image
# further attribute allow to crop parts of the tiling (the Poincare disk has radius r=100, centered on the origin)
canvas = SVGCanvas(width=400, height=400, radius=50, center=(50, -50))

# Add out-circles for each polygon
radius = outradius_regular_polygon(5,4)
circles = [HyperbolicCircle(z0=complex(T[i][0]), R=radius, lw=0.4) for i in range(len(T))]
canvas.add(*circles)

# Render and display
draw_svg(canvas.render())

## Basic Example: Attributes
Typical SVG arguments can be passed such as demonstrated below


In [ ]:
from hypertiling import HyperbolicTiling
from hypertiling.graphics.svg import *
import numpy as np

# Create tiling
T = HyperbolicTiling(3,8,5)
T.rotate(np.deg2rad(250))

pastel_colors = [
    "#96ceb4", 
    "#ffeead", 
    "#ff6f69", 
    "#ffcc5c",
    "#88d8b0",
    "#f7b2b2",
    "#4c8da8",
    "#f59670"
]

canvas = SVGCanvas()

# Tiling with vibrant pastels and white edges
facecolors = [pastel_colors[i % len(pastel_colors)] for i in range(len(T))]
canvas.add(Tiling(T, facecolors=facecolors, edgecolor="#FFFFFF", lw=1.2, skip_first=True))

draw_svg(canvas.render())

## Example: Horocycles and Fluent API
Here we demonstrate how elements (in this case horocycles can be modified using setter functions

In [ ]:
T = HyperbolicTiling(7,3,6)
canvas = SVGCanvas()

canvas.add(Tiling(T, facecolors="white", lw=0.4, skip_first=True, opacity=0.5))
canvas.add(UnitCircle(edgecolor="#001f3f", lw=1))

for dist in np.linspace(1,5,20):
    # use opacity
    horo = Horocycle(z1=np.exp(1j *  dist * np.pi / 4), R=0.3, edgecolor="#ff6f69", opacity=0.8)
    # use fluent API to modify properties
    horo.set_distance(dist).set_fill("#fff3c4").set_linewidth(2/dist)
    canvas.add(horo)

draw_svg(canvas.render())

The API allows to pass aribirtray keyword arguments, which will be forwarded to the SVG path. In this example, we use, e.g. stroke_dasharray

In [ ]:
canvas = SVGCanvas()

T = HyperbolicTiling(8,3,6)
canvas.add(Tiling(T, facecolors="#505050", edgecolor="#808080", lw=0.4, skip_first=True))
canvas.add(UnitCircle(edgecolor="#606060", lw=1))

# horocycle
horo = Horocycle(z1=1, R=1, edgecolor="#f59670", opacity=0.8, lw=1.5, stroke_dasharray="2,2")
canvas.add(horo)

# geodesic
g1 = Geodesic(complex(0.1, 0.4), complex(0.8, 0.9), edgecolor="#ffeead", lw=2)
g1.set_endpoints(1, complex(0.1, 0.4)) # modify endpoints
canvas.add(g1)

# more geodesics
for i in np.linspace(0,3,8):
    g1 = Geodesic(1, np.exp(-1j*i), edgecolor="#88d8b0", lw=1)
    canvas.add(g1)

# draw canvas
draw_svg(canvas.render())

Python print can be used to get information about the content of a canvas

In [ ]:
canvas = SVGCanvas()
canvas.add(UnitCircle())
canvas.add(Geodesic(0+0j, 0.5+0.5j))
canvas.add(HyperbolicCircle(0.3+0.2j, 0.4))

print(canvas.elements)
# Output: [UnitCircle(r=1.0, center=(0, 0)), Geodesic(0.00+0.00j → 0.50+0.50j), HyperbolicCircle(center=0.30+0.20j, R=0.40)]

print(canvas.elements[1])
# Output: Geodesic(0.00+0.00j → 0.50+0.50j)

## Example: Psychedelic

Concentric circles

In [ ]:
from hypertiling import HyperbolicTiling
from hypertiling.util import dual_edge_length_geodesic, edge_length_geodesic
from hypertiling.graphics.svg import *
p,q = 3,7
T = HyperbolicTiling(p,q,6)

In [ ]:
canvas = SVGCanvas()

dual_length = dual_edge_length_geodesic(p,q)

for i in range(len(T)):
    center = complex(T[i][0])
    
    # five concentric circles with decreasing line thickness
    for factor, thickness in [(0.5, 0.5), (0.4, 0.4), (0.3, 0.3), (0.2, 0.2), (0.1, 0.1)]:
        canvas.add(HyperbolicCircle(z0=center,R=factor * dual_length, lw=thickness))

draw_svg(canvas.render())

random colored triangular tiling

In [ ]:
import random

T = HyperbolicTiling(7,3,2,25, kernel="GRCT")

canvas = SVGCanvas()

for cell in T:
    center_dist = np.abs(cell[0])
    if center_dist < 0.98:
        canvas.add(Polygon(cell, opacity=random.random(), lw=0.3).set_fill("#202020").set_edgecolor("white"))

draw_svg(canvas.render())

## Example: SVG embedding

The SVGGroup element can be used to add arbitrary SVG paths to the canvas

In [ ]:
import numpy as np
from hypertiling import HyperbolicTiling
from hypertiling.graphics.svg import Tiling, UnitCircle, SVGCanvas, SVGGroup, draw_svg

# Create and rotate tiling
T = HyperbolicTiling(5, 4, 5)
T.rotate(2.2)

# Create SVG canvas
canvas = SVGCanvas()

# Draw boundary and tiling
canvas.add(UnitCircle(lw=0.5))
canvas.add(Tiling(T, lw=0.4, edgecolor="grey"))

# the svg object:
# first line sets the reference rotation
# second line the reference position
longhorn = '''
<g fill="#bf5700" transform="rotate(54,0,0)">
<g fill="#bf5700" transform="translate(122 40)">
<path d="m0 0c-9.484 2.578-20.969 0.665-30.204-2.164-17.64-5.659-32.699-17.641-49.091-26.211-8.655-3.661-20.555-4.41-28.542 0.583-1.913 1.358-5.38 0.388-5.38 0.388-1.109 0.831-2.352 1.148-3.856 0.611-3.994-2.914-8.155 1.58-12.396-0.75-4.242 2.33-8.402-2.164-12.396 0.75-1.505 0.537-2.747 0.22-3.856-0.611 0 0-3.469 0.97-5.382-0.388-7.986-4.993-19.885-4.244-28.539-0.583-16.394 8.57-31.452 20.552-49.092 26.211-9.236 2.829-20.72 4.742-30.205 2.164-1.248-0.832-2.664-2.331-2.331-4.161 0.666-1.082 1.333-2.58 2.747-2.83 33.95 5.076 52.254-28.623 80.712-37.526 0.055-0.361-1.609-1.747-3.413-2.718-4.021-2.22-9.985-1.359-10.4-6.851 1.081-3.413 5.241-4.661 8.237-5.991 7.073-3.997 15.562-0.668 21.967 2.578 0.748 0.75 1.997 0.334 2.578-0.416-0.499-15.395 11.234-25.63 13.397-39.941 1.249-6.657-2.912-11.564-3.411-17.806-0.055-2.303 1.747-0.778 2.718-8.404 0.417-1.387 3.884-5.271 7.018-6.906 2.893-1.14 6.246-1.472 9.651-1.383 3.404-0.089 6.757 0.243 9.65 1.383 3.134 1.635 6.602 5.519 7.015 6.906 0.974 7.626 2.776 6.101 2.721 8.404-0.5 6.242-4.66 11.149-3.412 17.806 2.164 14.311 13.896 24.546 13.396 39.941 0.584 0.75 1.833 1.166 2.581 0.416 6.405-3.246 14.892-6.575 21.967-2.578 2.995 1.33 7.155 2.578 8.237 5.991-0.417 5.492-6.381 4.631-10.4 6.851-1.805 0.971-3.469 2.357-3.415 2.718 28.459 8.903 46.765 42.602 80.714 37.526 1.414 0.25 2.08 1.748 2.744 2.83 0.323 1.828-1.09 3.327-2.338 4.159" fill="#bf5700"/></g></g>
'''

# Iterate over tiling
for i in range(0, len(T)):
    # extract center position of cell
    center = complex(T[i][0])
    
    # extract orientation of cell (depends on tiling construction kernel!)
    first_vertex = complex(T[i][1])
    rotation = - np.angle(first_vertex - center)
    
    # draw svg object
    scale = 0.18 * (1 - abs(center)**2) # hyperbolic scale
    group = SVGGroup(longhorn, center=center, scale=scale, rotation=rotation)
    canvas.add(group)

# Render and display
draw_svg(canvas.render())